Download http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz and unzip into './Data/IMDB/files/aclImdb'

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from modules import data_loaders, learning
from modules.models import ClassificationNet

In [7]:
# n_hidden = 128
n_hidden = 3
n_emb = 300
seq_len = 200
maxlen = 200
save_fq = 200

batch_size = 128
index_from = 3
vocab_size = 20000
learning_rate = 0.0005
num_epochs = 1000

# args = sys.argv[1:]
# config = args[0]

config = 'LCLCCL'

In [13]:
base_data_path = 'Data/IMDB/files/'
paths_train = [base_data_path + "aclImdb/train/pos", base_data_path + "aclImdb/train/neg"]
paths_test = [base_data_path + "aclImdb/test/pos", base_data_path + "aclImdb/test/neg"]
labels = [1, 0]

(X_train, y_train, mask_train), (X_val, y_val, mask_val), (X_test, y_test, mask_test), vocab = data_loaders.process_imdb(paths_train, 
                                                                                                            paths_test, 
                                                                                                            labels,
                                                                                                            num_words=vocab_size,
                                                                                                            maxlen=maxlen)


In [17]:
train_data = data_loaders.Reviews(X_train, 
                                  y_train, 
                                  batch_size,
                                  mask=mask_train,
                                  shuffle=True)
val_data = data_loaders.Reviews(X_val, y_val, batch_size, mask=mask_val)
test_data = data_loaders.Reviews(X_test, y_test, batch_size, mask=mask_test)

train_loader = data_loaders.ReviewsLoader(train_data)
valid_loader = data_loaders.ReviewsLoader(val_data)
test_loader = data_loaders.ReviewsLoader(test_data)


In [18]:
def accuracy(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    pred_labels = (prediction >= 0.5).to(target.dtype)
    return (pred_labels == target).float().mean()

def loss_function(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x = x.view(-1)
    x = x.flatten().clamp(1e-6, 1 - 1e-6).float()
    y = y.view(-1).float()
    return nn.functional.binary_cross_entropy(x, y)

In [19]:
class_net = ClassificationNet(vocab_size=vocab_size+index_from, 
                              n_emb=n_emb, 
                              n_hidden=n_hidden, 
                              num_classes=1, 
                              config=config)

optimizer = optim.Adam(class_net.parameters(), lr=learning_rate)

lm_trainer = learning.LMTrainer(model=class_net,
                                optimizer=optimizer,
                                criterion=loss_function,
                                val_criterion=accuracy,
                                num_epochs=num_epochs,
                                train_loader=train_loader,
                                valid_loader=valid_loader,
                                test_loader=test_loader)

lm_trainer.train()


epoch 0001/1000 | batch 0000/167 | base_loss 0.7118 | total_loss -144.9331 | tokens 128 


KeyboardInterrupt: 

profiling

In [ ]:
import torch.profiler as profiler


def test_with_loader(model, loader):
    model.train()
    # -------------------
    # профилируем одну эпоху
    # -------------------
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CPU],
        record_shapes=True,
        with_stack=True
    ) as prof:
        for step, (x, y, _) in enumerate(loader):
            with profiler.record_function("model_inference"):
                logits = model(x)
            if step >= 2:
                break
    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))


model = ClassificationNet(vocab_size=vocab_size+index_from, 
                              n_emb=n_emb, 
                              n_hidden=n_hidden, 
                              num_classes=1, 
                              config=config)

test_with_loader(model, train_loader)


-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
        model_inference        26.97%     221.827ms        98.59%     811.013ms     270.338ms             3  
              aten::mul        15.35%     126.249ms        18.37%     151.099ms      16.705us          9045  
       aten::randn_like         0.05%     440.400us        12.90%     106.145ms      17.691ms             6  
          aten::normal_        12.84%     105.644ms        12.84%     105.644ms       8.804ms            12  
              aten::add         9.15%      75.271ms        11.53%      94.879ms      13.090us          7248  
            aten::slice         6.84%      56.252ms         7.67%      63.123ms       6.541us          9651  
          